# Softmax and numerical stability

This CPU lesson builds the reference operation step by step. It does not require or claim CUDA execution.

In [ ]:
import torch

from cuda_attention.reference import stable_softmax

torch.manual_seed(0)
scores = torch.randn(2, 4, dtype=torch.float32)
assert scores.shape == (2, 4)
scores

## Why direct exponentiation can fail

FP32 cannot represent arbitrarily large exponentials. A stable implementation shifts every row by its maximum before exponentiation.

In [ ]:
large_scores = torch.tensor([[1000.0, 999.0, 998.0]])
naive_exponentials = torch.exp(large_scores)
shifted_scores = large_scores - large_scores.amax(dim=-1, keepdim=True)
stable_exponentials = torch.exp(shifted_scores)

assert torch.isinf(naive_exponentials).all()
assert torch.isfinite(stable_exponentials).all()
shifted_scores, stable_exponentials

Subtracting a constant does not change softmax because `exp(-m)` multiplies every numerator and the denominator, where it cancels. The largest shifted logit is zero, so its exponential is one.

In [ ]:
reference = stable_softmax(large_scores)
pytorch = torch.softmax(large_scores, dim=-1)

torch.testing.assert_close(reference, pytorch)
torch.testing.assert_close(reference.sum(dim=-1), torch.ones(1))
assert torch.isfinite(reference).all()
reference

## Exercises

1. Predict the probabilities for four equal logits before running code.
2. Change the reduction dimension of a 3-D tensor and explain which values compete.
3. Explain why row sums may be approximately, rather than bitwise, equal to one.

TODO(student): Explain maximum subtraction in your own words.

TODO(student): Record any question that remains after running the cells.